# 1. PDF 로드 산출물 전처리 목적과 페이지 보존 원칙

`03_load_pdf_json`의 유니크 PDF 10종을 페이지 단위로 정제한다. 청킹·병합·자동 dedup은 수행하지 않으며, 빈 표지·글자 뒤섞임·제목 구분 페이지만 근거와 함께 제외한다.

## 2. 경로·문서 레지스트리 설정

선택할 로드 JSON, 대응 원본 PDF, 문서 메타데이터와 스캔본의 실측 오프셋을 명시한다.

In [1]:
from __future__ import annotations

import hashlib
import html
import json
import re
import shutil
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

from pypdf import PdfReader


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / '.git').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'data' / 'legal_api_v2'
LOAD_DIR = DATA_ROOT / '03_load_pdf_json'
RAW_PDF_DIR = DATA_ROOT / '01_raw_pdf'
OUTPUT_ROOT = DATA_ROOT / '05_processed_pdf_json'
TEXT_OUTPUT_DIR = OUTPUT_ROOT / 'text'
VALIDATION_PATH = OUTPUT_ROOT / 'preprocess_validation.json'
JSONL_MAX_BYTES = 50 * 1024 * 1024

DOC_REGISTRY = {
    'mediation_casebook_2021': {'load_json': '2021_[한국부동산원]_주택·상가건물_임대차분쟁조정사례집_processed.json', 'raw_pdf': '2021_[한국부동산원] 주택·상가건물 임대차분쟁조정사례집.pdf', 'source_type': 'mediation_casebook', 'doc_title': '2021 주택·상가건물 임대차분쟁조정사례집', 'source_org': '한국부동산원', 'doc_year': '2021', 'issue': '분쟁조정', 'fixed_offset': 4},
    'mediation_casebook_2022': {'load_json': '[국토교통부]_2022_주택임대차분쟁조정사례집_processed.json', 'raw_pdf': '[국토교통부] 2022 주택임대차분쟁조정사례집.pdf', 'source_type': 'mediation_casebook', 'doc_title': '2022 주택임대차분쟁조정사례집', 'source_org': '국토교통부', 'doc_year': '2022', 'issue': '분쟁조정', 'fixed_offset': 3},
    'mediation_casebook_2023': {'load_json': '2023_주택·상가건물_임대차분쟁조정사례집(한국부동산원)_processed.json', 'raw_pdf': '2023 주택·상가건물 임대차분쟁조정사례집(한국부동산원).pdf', 'source_type': 'mediation_casebook', 'doc_title': '2023 주택·상가건물 임대차분쟁조정사례집', 'source_org': '한국부동산원', 'doc_year': '2023', 'issue': '분쟁조정', 'fixed_offset': 1},
    'counsel_casebook_2024': {'load_json': '2024_주택·상가건물_임대차분쟁조정_사례집_processed.json', 'raw_pdf': '2024_주택·상가건물 임대차분쟁조정 사례집.pdf', 'source_type': 'counsel_casebook', 'doc_title': '2024 주택임대차 상담사례집', 'source_org': '국토교통부·한국부동산원', 'doc_year': '2024', 'issue': '임대차상담', 'fixed_offset': None},
    'counsel_casebook_2025': {'load_json': '2025_주택 및 상가건물 임대차 상담사례집_국토교통부_한국부동산원_(최종_배포)_processed.json', 'raw_pdf': '2025_주택 및 상가건물 임대차 상담사례집_국토교통부_한국부동산원_(최종_배포).pdf', 'source_type': 'counsel_casebook', 'doc_title': '2025 주택 및 상가건물 임대차 상담사례집', 'source_org': '국토교통부·한국부동산원', 'doc_year': '2025', 'issue': '임대차상담', 'fixed_offset': None},
    'standard_contract_202310': {'load_json': '2022_개정_주택임대차_표준계약서(개정전)_processed.json', 'raw_pdf': '2022_개정_주택임대차 표준계약서(개정전).pdf', 'source_type': 'standard_contract', 'doc_title': '주택임대차표준계약서 (2023.10 개정)', 'source_org': '법무부·국토교통부', 'doc_year': '2023', 'issue': '계약체결', 'fixed_offset': None},
    'guide_hlpa_guidebook_2020': {'load_json': '주택임대차보호법 가이드북_20200731_개정판_processed.json', 'raw_pdf': '주택임대차보호법 가이드북_20200731_개정판.pdf', 'source_type': 'guide', 'doc_title': '주택임대차보호법 가이드북 (2020.7.31 개정판)', 'source_org': '국토교통부·법무부', 'doc_year': '2020', 'issue': '법령해설', 'fixed_offset': None},
    'guide_legal_procedure_seoul': {'load_json': '소송등 법적절차 안내문(서울시 전월세종합지원센터)_processed.json', 'raw_pdf': '소송등 법적절차 안내문(서울시 전월세종합지원센터).pdf', 'source_type': 'guide', 'doc_title': '소송등 법적절차 안내문', 'source_org': '서울시 전월세종합지원센터', 'doc_year': '', 'issue': '분쟁대응', 'fixed_offset': None},
    'guide_jeonse_fraud_seoul': {'load_json': '전월세종합지원센터_챗봇서비스_전세사기 상담 지원_서울특별시_processed.json', 'raw_pdf': '전월세종합지원센터_챗봇서비스_전세사기 상담 지원_서울특별시.pdf', 'source_type': 'guide', 'doc_title': '전세사기 상담 지원 안내', 'source_org': '서울특별시', 'doc_year': '', 'issue': '전세사기', 'fixed_offset': None},
    'guide_contract_checklist_2026': {'load_json': '주택임대차전월세계약시 주요 확인사항_20260313_processed.json', 'raw_pdf': '주택임대차전월세계약시 주요 확인사항_20260313.pdf', 'source_type': 'guide', 'doc_title': '주택임대차(전월세) 계약 시 주요 확인사항', 'source_org': '', 'doc_year': '2026', 'issue': '계약체결', 'fixed_offset': None},
}

EXCLUDED_DUPLICATES = [
    '2022_주택·상가건물_임대차분쟁조정_사례집_processed.json',
    '2023_주택·상가건물_임대차분쟁조정_사례집_processed.json',
    '20231006_개정_주택임대차 표준계약서_특약사항 포함(개정후)_processed.json',
    '2024_주택·상가건물 임대차분쟁조정 사례집_processed.json',
    '2024_주택임대차 상담사례집(최종_배포)_processed.json',
    '2025_주택·상가건물 임대차분쟁조정 사례집_processed.json',
]

assert LOAD_DIR.is_dir() and RAW_PDF_DIR.is_dir()
assert len(DOC_REGISTRY) == 10 and len(set(DOC_REGISTRY)) == 10
selected_loads = {item['load_json'] for item in DOC_REGISTRY.values()}
all_loads = {path.name for path in LOAD_DIR.glob('*.json')}
assert all_loads == selected_loads | set(EXCLUDED_DUPLICATES)
print(f'입력 로드 JSON: {len(all_loads)}개 / 선택: {len(selected_loads)}개 / 중복 제외: {len(EXCLUDED_DUPLICATES)}개')


입력 로드 JSON: 16개 / 선택: 10개 / 중복 제외: 6개


## 3. 입력 inventory·schema·원본 PDF md5 검증

신포맷은 로드 metadata의 md5를 원본과 전건 대조하고, 구포맷은 원본에서 계산한 값을 이후 metadata에 사용한다.

In [2]:
def file_md5(path: Path) -> str:
    digest = hashlib.md5()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


input_records_by_slug: dict[str, list[dict[str, Any]]] = {}
pdf_md5_by_slug: dict[str, str] = {}
legacy_by_slug: dict[str, bool] = {}
inventory: dict[str, dict[str, Any]] = {}

for slug, cfg in DOC_REGISTRY.items():
    load_path = LOAD_DIR / cfg['load_json']
    raw_path = RAW_PDF_DIR / cfg['raw_pdf']
    assert load_path.is_file() and raw_path.is_file(), (slug, load_path, raw_path)
    records = json.loads(load_path.read_text(encoding='utf-8'))
    assert isinstance(records, list) and records
    pages = [row['metadata']['page'] for row in records]
    assert pages == sorted(pages) and len(pages) == len(set(pages))
    assert all(isinstance(row.get('page_content'), str) and isinstance(row.get('metadata'), dict) for row in records)
    total_pages_values = {row['metadata'].get('total_pages') for row in records}
    assert len(total_pages_values) == 1 and next(iter(total_pages_values)) >= max(pages) + 1
    digest = file_md5(raw_path)
    is_legacy = all('method' not in row['metadata'] for row in records)
    if is_legacy:
        assert all('md5' not in row['metadata'] for row in records)
    else:
        assert all(row['metadata'].get('method') in {'vision', 'text'} for row in records)
        assert all(row['metadata'].get('md5') == digest for row in records), slug
    input_records_by_slug[slug] = records
    pdf_md5_by_slug[slug] = digest
    legacy_by_slug[slug] = is_legacy
    inventory[slug] = {'records': len(records), 'total_pages': next(iter(total_pages_values)), 'format': 'legacy' if is_legacy else 'new', 'pdf_md5': digest}

print(json.dumps(inventory, ensure_ascii=False, indent=2))


{
  "mediation_casebook_2021": {
    "records": 195,
    "total_pages": 195,
    "format": "new",
    "pdf_md5": "5d8000aeea6a349ac022d04601a1ac51"
  },
  "mediation_casebook_2022": {
    "records": 152,
    "total_pages": 152,
    "format": "new",
    "pdf_md5": "eb0834b6254fdece2373ffb8e2e207aa"
  },
  "mediation_casebook_2023": {
    "records": 96,
    "total_pages": 96,
    "format": "new",
    "pdf_md5": "53583fbce752c57fdbcf0350b848b2eb"
  },
  "counsel_casebook_2024": {
    "records": 113,
    "total_pages": 113,
    "format": "new",
    "pdf_md5": "8d169866cd6b2b7ab8945a067e001941"
  },
  "counsel_casebook_2025": {
    "records": 121,
    "total_pages": 126,
    "format": "legacy",
    "pdf_md5": "8c1e31d0082b385a03a3d67b06c7310f"
  },
  "standard_contract_202310": {
    "records": 5,
    "total_pages": 5,
    "format": "new",
    "pdf_md5": "fadb1d03b9b728b3bdb81a07d6729ef5"
  },
  "guide_hlpa_guidebook_2020": {
    "records": 131,
    "total_pages": 134,
    "format": "legacy

## 4. 의미 보존형 텍스트·HTML 전처리 함수

02 노트북의 함수와 정규식을 복사해 사용하고, PDF 로더의 주석 마커만 앞단에서 제거한다.

In [3]:
TABLE_VERTICAL_RE = re.compile(r'[│┃║]')
TABLE_HORIZONTAL_RE = re.compile(r'[─━═┄┅┈┉╌╍]+')
TABLE_JUNCTION_RE = re.compile(r'[┌┐└┘├┤┬┴┼┏┓┗┛┣┫┳┻╋╔╗╚╝╠╣╦╩╬]')
TABLE_LAYOUT_ONLY_RE = re.compile(r'^[\s│┃║─━═┄┅┈┉╌╍┌┐└┘├┤┬┴┼┏┓┗┛┣┫┳┻╋╔╗╚╝╠╣╦╩╬|+_-]+$')


def flatten_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return '\n'.join(part for item in value if (part := flatten_text(item)))
    if isinstance(value, dict):
        if value.get('content') not in (None, ''):
            return flatten_text(value['content'])
        return '\n'.join(part for item in value.values() if (part := flatten_text(item)))
    return str(value)


def normalize_meaning_preserving_lines(value: Any, *, table: bool = False) -> str:
    text = unicodedata.normalize('NFC', html.unescape(flatten_text(value)))
    text = text.replace('\r\n', '\n').replace('\r', '\n').replace('\u00a0', ' ')
    lines: list[str] = []
    for raw_line in text.split('\n'):
        line = raw_line
        if table:
            if TABLE_LAYOUT_ONLY_RE.fullmatch(line):
                continue
            line = TABLE_VERTICAL_RE.sub(' | ', line)
            line = TABLE_HORIZONTAL_RE.sub(' ', line)
            line = TABLE_JUNCTION_RE.sub(' ', line)
            line = re.sub(r'\s*\|\s*', ' | ', line)
            line = re.sub(r'(?:\s*\|\s*){2,}', ' | ', line)
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if table:
            line = line.strip(' |').strip()
        lines.append(line)
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()


IMAGE_TAG_RE = re.compile(r'<img\b[^>]*>', re.IGNORECASE)
IMAGE_CLOSE_TAG_RE = re.compile(r'</\s*img\s*>', re.IGNORECASE)
MARKDOWN_IMAGE_RE = re.compile(r'!\[([^\]]*)\]\(([^)\s]+)(?:\s+[\x22\x27][^\x22\x27]*[\x22\x27])?\)')
BLOCK_TAG_RE = re.compile(r'</?(?:p|div|li|tr|table|thead|tbody|tfoot|ul|ol|h[1-6])\b[^>]*>', re.IGNORECASE)
BREAK_TAG_RE = re.compile(r'<br\s*/?>', re.IGNORECASE)
CELL_TAG_RE = re.compile(r'</?(?:td|th)\b[^>]*>', re.IGNORECASE)
INLINE_TAG_RE = re.compile(r'</?(?:span|font|b|strong|em|i|u|small|sup|sub|a)\b[^>]*>', re.IGNORECASE)
ALLOWLIST_CLOSING_TAG_RE = re.compile(r'</\s*(?:img|br|p|div|li|tr|table|thead|tbody|tfoot|ul|ol|h[1-6]|td|th|span|font|b|strong|em|i|u|small|sup|sub|a)\b[^>]*>', re.IGNORECASE)
HTML_COMMENT_RE = re.compile(r'<!--.*?-->', re.DOTALL)
COVER_MARKER_RE = re.compile(r'<!--\s*\(빈/표지 페이지\)\s*-->')
PICTURE_MARKER_RE = re.compile(r'<!--\s*(?:Start|End) of picture text\s*-->', re.IGNORECASE)


def preprocess_text(value: Any, *, table: bool = False) -> str:
    text = flatten_text(value)
    text = IMAGE_TAG_RE.sub('', text)
    text = IMAGE_CLOSE_TAG_RE.sub('', text)
    text = MARKDOWN_IMAGE_RE.sub('', text)
    text = BREAK_TAG_RE.sub('\n', text)
    text = CELL_TAG_RE.sub(' | ', text)
    text = BLOCK_TAG_RE.sub('\n', text)
    text = INLINE_TAG_RE.sub('', text)
    return normalize_meaning_preserving_lines(text, table=table)


def basic_pdf_cleanup(value: Any) -> tuple[str, bool]:
    raw = flatten_text(value)
    had_cover_marker = bool(COVER_MARKER_RE.search(raw))
    raw = PICTURE_MARKER_RE.sub('', raw)
    raw = HTML_COMMENT_RE.sub('', raw)
    return preprocess_text(raw), had_cover_marker


## 5. `book_page` 오프셋 산정

스캔본 3종은 실측 상수를 사용한다. 나머지 7종은 pypdf 경계부의 인쇄 번호를 추출해 같은 오프셋이 후보 페이지의 80% 이상에서 관측될 때만 채택한다.

In [4]:
BOUNDARY_NUMBER_RES = [
    re.compile(r'^\s*[-–—]?\s*(\d{1,3})(?:\s*/\s*\d{1,3})?\s*[-–—]?\s*$'),
    re.compile(r'^\s*[-–—]?\s*(\d{1,3})(?:\s*/\s*\d{1,3})?\s*[-–—]?\s+'),
    re.compile(r'(?:^|\s)[-–—]?\s*(\d{1,3})\s*[-–—]?\s*$'),
    re.compile(r'문의해\s*주십시오\.\s*(\d{1,3})\s*$'),
]


def infer_book_page_offset(raw_pdf: Path) -> dict[str, Any]:
    reader = PdfReader(raw_pdf)
    candidates_by_page: dict[int, set[int]] = {}
    for page_index, page in enumerate(reader.pages):
        lines = [line.strip() for line in (page.extract_text() or '').splitlines() if line.strip()]
        candidates: set[int] = set()
        for line in [*lines[:5], *lines[-5:]]:
            for pattern in BOUNDARY_NUMBER_RES:
                match = pattern.search(line)
                if not match:
                    continue
                printed = int(match.group(1))
                if 1 <= printed <= len(reader.pages) + 20 and abs(page_index - printed) <= 20:
                    candidates.add(printed)
        if candidates:
            candidates_by_page[page_index] = candidates
    votes = Counter(page_index - printed for page_index, values in candidates_by_page.items() for printed in values)
    if not votes:
        return {'offset': None, 'method': 'pypdf_boundary_regex', 'candidate_pages': 0, 'matched_pages': 0, 'agreement': 0.0, 'reason': '경계부에서 인쇄 페이지 번호 후보를 찾지 못함'}
    offset, _ = votes.most_common(1)[0]
    matched_pages = sum(1 for page_index, values in candidates_by_page.items() if page_index - offset in values)
    agreement = matched_pages / len(candidates_by_page)
    accepted = matched_pages >= 4 and agreement >= 0.80
    return {
        'offset': offset if accepted else None,
        'method': 'pypdf_boundary_regex_80pct',
        'candidate_pages': len(candidates_by_page),
        'matched_pages': matched_pages,
        'agreement': round(agreement, 6),
        'reason': '' if accepted else '동일 오프셋이 후보 본문 페이지의 80% 이상에서 확인되지 않음',
    }


book_page_info: dict[str, dict[str, Any]] = {}
for slug, cfg in DOC_REGISTRY.items():
    if cfg['fixed_offset'] is not None:
        book_page_info[slug] = {'offset': cfg['fixed_offset'], 'method': 'measured_constant_scan', 'candidate_pages': None, 'matched_pages': None, 'agreement': 1.0, 'reason': ''}
    else:
        book_page_info[slug] = infer_book_page_offset(RAW_PDF_DIR / cfg['raw_pdf'])

print(json.dumps(book_page_info, ensure_ascii=False, indent=2))


{
  "mediation_casebook_2021": {
    "offset": 4,
    "method": "measured_constant_scan",
    "candidate_pages": null,
    "matched_pages": null,
    "agreement": 1.0,
    "reason": ""
  },
  "mediation_casebook_2022": {
    "offset": 3,
    "method": "measured_constant_scan",
    "candidate_pages": null,
    "matched_pages": null,
    "agreement": 1.0,
    "reason": ""
  },
  "mediation_casebook_2023": {
    "offset": 1,
    "method": "measured_constant_scan",
    "candidate_pages": null,
    "matched_pages": null,
    "agreement": 1.0,
    "reason": ""
  },
  "counsel_casebook_2024": {
    "offset": 0,
    "method": "pypdf_boundary_regex_80pct",
    "candidate_pages": 82,
    "matched_pages": 79,
    "agreement": 0.963415,
    "reason": ""
  },
  "counsel_casebook_2025": {
    "offset": 1,
    "method": "pypdf_boundary_regex_80pct",
    "candidate_pages": 108,
    "matched_pages": 105,
    "agreement": 0.972222,
    "reason": ""
  },
  "standard_contract_202310": {
    "offset": -1,


## 6. 구포맷 반복 머리말·꼬리말 분석과 scrambled/divider 판정

숫자를 `#`로 마스킹할 수 있는 경계 패턴을 문서별로 집계하고 4페이지 이상 반복된 규칙만 적용한다. 제거 전후 70% 이상 소실 여부는 별도 검증한다.

In [5]:
LEGACY_BOUNDARY_RULES: dict[str, list[tuple[str, str]]] = {
    'counsel_casebook_2025': [
        ('상담사례집 시작 러닝타이틀', r'^주택 및 상가건물 임대차 상담사례집\s+'),
        ('상담사례집 끝 인쇄번호·러닝타이틀', r'\s+\d{1,3}\s+주택 및 상가건물 임대차 상담사례집$'),
        ('상담사례집 탭 내비게이션', r'제1장 제도소개 제2장 주택임대차 상담사례(?:\[[^\]]+\])? 제3장 상가(?:건물)?임대차 상담사례 부록\s*'),
    ],
    'guide_hlpa_guidebook_2020': [
        ('가이드북 시작 러닝헤더 변형', r'^(?:\d{1,3}\s+)?2020\. 7\. 31\. 개정 주택임대차보호법 가이드북\s+\d{1,3}(?:\s+국토교통부-법무부 발간 주택임대차보호법 해설집)?\s*'),
        ('만화 시작 인쇄번호·러닝헤더', r'^\d{1,3}\s+만화로 보는 주택임(?:대차|차)보호법\s+[‘\x22\x27](?:세입자|집주인)편[’\x22\x27]\s*'),
        ('서울시 상담 시작 러닝헤더', r'^\d{1,3}\s+서울시 주요상담 사례\s+\d{1,3}\s+'),
        ('부록 시작 인쇄번호·러닝헤더', r'^\d{1,3}\s+부록\s+'),
        ('뒤섞임·겹자 러닝헤더', r'^\d{4}\s+(?:만서화울로시 보주는요 상주담택 임사대례차보호법|만주화택로임 대보차는분 주쟁택조임정대 사차례보호법)\s+[‘\x22\x27]세입자편[’\x22\x27](?:\s+\d{1,3})?\s*'),
    ],
    'guide_legal_procedure_seoul': [
        ('서울시 견본서식 끝 안내문', r'\s*본 서식은 서울시 주택정책과에서 작성한 견본 문서이므로, 신청인의 구체적인 신청 사유에 맞게 수정하여 사용하시기 바랍니다\(문의사항: 02-2133-1200~1208\(3\)\)\.$'),
    ],
    'guide_jeonse_fraud_seoul': [
        ('보도자료 인쇄 페이지 번호', r'(?:^|\s+)-\s*\d{1,3}\s*-(?=\s+|$)'),
    ],
    'guide_contract_checklist_2026': [
        ('주요 확인사항 끝 문의·인쇄번호', r'\s*궁금한 사항은 서울시 전․월세 종합지원센터\(02-2133-1200\)로 문의해 주십시오\.\s*\d{1,3}(?=\s|$)'),
    ],
}


def masked_pattern(pattern: str) -> str:
    return re.sub(r'\\d\{1,3\}', '#', pattern)


active_boundary_rules: dict[str, list[tuple[str, re.Pattern[str], int]]] = defaultdict(list)
removed_pattern_report: dict[str, list[str]] = defaultdict(list)
for slug, rules in LEGACY_BOUNDARY_RULES.items():
    assert legacy_by_slug[slug]
    cleaned_pages = [basic_pdf_cleanup(row['page_content'])[0] for row in input_records_by_slug[slug]]
    for label, expression in rules:
        pattern = re.compile(expression)
        hit_pages = sum(bool(pattern.search(text)) for text in cleaned_pages)
        if hit_pages >= 4:
            active_boundary_rules[slug].append((label, pattern, hit_pages))
            removed_pattern_report[slug].append(f'{label} (숫자 마스킹: {masked_pattern(expression)}, {hit_pages}페이지)')


def compact_letters(value: str) -> str:
    return re.sub(r'[^0-9A-Za-z가-힣]', '', value).lower()


SCRAMBLED_COMIC_PAGES = {6, 7, 38, 40, 41}
LEGAL_TEXT_INTERLEAVED_PAGES = set(range(100, 113))


def is_scrambled(slug: str, page_index: int, text_after_header_cleanup: str) -> bool:
    if slug != 'guide_hlpa_guidebook_2020':
        return False
    assert text_after_header_cleanup.strip()
    return page_index in SCRAMBLED_COMIC_PAGES or page_index in LEGAL_TEXT_INTERLEAVED_PAGES


def is_divider(text: str, title: str) -> bool:
    if re.fullmatch(r'제\d+장\s+.{0,40}(?:상담사례|제도소개)', text.strip()):
        return True
    compact = compact_letters(text)
    compact_title = compact_letters(title)
    compact = re.sub(r'^(?:content)+', '', compact)
    if re.sub(r'\d+', '', compact) in {'부록', '부록부록'}:
        return True
    if not compact or len(compact) > max(90, len(compact_title) * 2):
        return False
    comparable = re.sub(r'\d+|개정판?', '', compact)
    comparable_title = re.sub(r'\d+|개정판?', '', compact_title)
    return compact in compact_title or compact_title in compact or comparable in comparable_title or comparable_title in comparable


def remove_repeated_boundaries(slug: str, text: str) -> tuple[str, list[str]]:
    current = text
    applied: list[str] = []
    for label, pattern, _ in active_boundary_rules.get(slug, []):
        updated, count = pattern.subn(' ', current)
        if count:
            current = updated
            applied.append(label)
    return normalize_meaning_preserving_lines(current), applied


STANDARD_CONTRACT_FOOTER_RE = re.compile(r'-\s*\d+\s*/\s*\d+\s*-')
OVERLAPPED_CHARACTER_RUN_RE = re.compile(r'(?:(?P<pair_char>[0-9A-Za-z가-힣()\[\]【】<>])(?P=pair_char)(?:\s+)?){3,}')
RUNAWAY_REPETITION_VALIDATION_RE = re.compile(r'(\S{1,6}\s+)\1{7,}')
RUNAWAY_REPETITION_COLLAPSE_RE = re.compile(r'(?P<unit>(?P<token>\S{1,6})\s+)(?P=unit){7,}(?:(?P=token)(?!\S))?')
GUIDE_RUNNING_HEADER_RESIDUAL_RE = re.compile(
    r'^(?:(?:\d{1,3}\s+)?2020\. 7\. 31\. 개정 주택임대차보호법 가이드북\s+\d{1,3}(?:\s|$)'
    r'|\d{1,4}\s+만화로 보는 주택임(?:대차|차)보호법'
    r'|\d{1,3}\s+서울시 주요상담 사례\s+\d{1,3}'
    r'|\d{1,3}\s+부록(?:\s|$)'
    r'|\d{4}\s+(?:만서화울로시|만주화택로임))'
)


def collapse_overlapped_character_runs(text: str) -> tuple[str, int]:
    def collapse(match: re.Match[str]) -> str:
        return re.sub(r'([0-9A-Za-z가-힣()\[\]【】<>])\1', r'\1', match.group(0))
    return OVERLAPPED_CHARACTER_RUN_RE.subn(collapse, text)


def collapse_runaway_repetitions(text: str) -> tuple[str, int]:
    def collapse(match: re.Match[str]) -> str:
        token = match.group('token')
        trailing_whitespace = re.search(r'\s+$', match.group(0))
        return f'{token} {token}' + (trailing_whitespace.group(0) if trailing_whitespace else '')

    collapsed, count = RUNAWAY_REPETITION_COLLAPSE_RE.subn(collapse, text)
    return (collapsed.rstrip() if count else text), count


def apply_document_specific_cleanup(slug: str, text: str) -> tuple[str, Counter[str]]:
    counts: Counter[str] = Counter()
    if slug == 'standard_contract_202310':
        text, count = STANDARD_CONTRACT_FOOTER_RE.subn(' ', text)
        counts['표준계약서 인쇄 푸터'] += count
    if slug == 'counsel_casebook_2025':
        text, count = collapse_overlapped_character_runs(text)
        counts['3쌍 이상 겹자 런'] += count
    return normalize_meaning_preserving_lines(text), +counts


DOTTED_LEADER_TOC_LINE_RE = re.compile(r'(?m)^.*·{4,}.*\d{1,3}\s*\|?\s*$')
TRAILING_BARE_PAGE_NUMBER_RE = re.compile(r'\n(\d{1,3})\s*$')
EXPECTED_TOC_PAGES = {('counsel_casebook_2024', 3)}


def is_dotted_leader_toc(text: str) -> bool:
    return len(DOTTED_LEADER_TOC_LINE_RE.findall(text)) >= 3


detected_toc_pages = {
    (slug, int(row['metadata']['page']))
    for slug, rows in input_records_by_slug.items()
    for row in rows
    if is_dotted_leader_toc(basic_pdf_cleanup(row['page_content'])[0])
}
assert detected_toc_pages == EXPECTED_TOC_PAGES, detected_toc_pages


MARKDOWN_TABLE_SEPARATOR_CANDIDATE_RE = re.compile(r'^\|[ \t|:-]+\|$')
COMPACT_MARKDOWN_TABLE_SEPARATOR_RE = re.compile(r'^\|(?::?-{3}:?\|)+$')
EXPECTED_MARKDOWN_TABLE_COUNT = 404


def is_blank_pipe_row(line: str) -> bool:
    return bool(MARKDOWN_TABLE_SEPARATOR_CANDIDATE_RE.fullmatch(line)) and '-' not in line and ':' not in line and any(character.isspace() for character in line)


def markdown_table_separator_indices(lines: list[str]) -> list[int]:
    indices: list[int] = []
    for index, line in enumerate(lines):
        if not MARKDOWN_TABLE_SEPARATOR_CANDIDATE_RE.fullmatch(line):
            continue
        # 연속 빈 파이프 행은 표 내부의 공백 데이터 행이며, 파이프만 있는 '|||'도 구분선이 아니다.
        if not any(character in line for character in '-: \t'):
            continue
        if is_blank_pipe_row(line) and ((index > 0 and is_blank_pipe_row(lines[index - 1])) or (index + 1 < len(lines) and is_blank_pipe_row(lines[index + 1]))):
            continue
        indices.append(index)
    return indices


def table_separator_signature(line: str) -> tuple[int, tuple[tuple[bool, bool], ...]]:
    cells = line[1:-1].split('|')
    alignment = tuple((cell.strip().startswith(':'), cell.strip().endswith(':')) for cell in cells)
    return len(cells), alignment


def compact_table_separator(line: str) -> str:
    cells = line[1:-1].split('|')
    compact_cells: list[str] = []
    for cell in cells:
        stripped = cell.strip()
        compact_cells.append((':' if stripped.startswith(':') else '') + '---' + (':' if stripped.endswith(':') else ''))
    return '|' + '|'.join(compact_cells) + '|'


def normalize_markdown_table_separators(text: str) -> tuple[str, list[tuple[int, int, tuple[tuple[bool, bool], ...]]]]:
    lines = text.split('\n')
    signatures: list[tuple[int, int, tuple[tuple[bool, bool], ...]]] = []
    for line_index in markdown_table_separator_indices(lines):
        column_count, alignment = table_separator_signature(lines[line_index])
        signatures.append((line_index, column_count, alignment))
        lines[line_index] = compact_table_separator(lines[line_index])
    return '\n'.join(lines), signatures

print(json.dumps(removed_pattern_report, ensure_ascii=False, indent=2))


{
  "counsel_casebook_2025": [
    "상담사례집 시작 러닝타이틀 (숫자 마스킹: ^주택 및 상가건물 임대차 상담사례집\\s+, 54페이지)",
    "상담사례집 끝 인쇄번호·러닝타이틀 (숫자 마스킹: \\s+#\\s+주택 및 상가건물 임대차 상담사례집$, 46페이지)",
    "상담사례집 탭 내비게이션 (숫자 마스킹: 제1장 제도소개 제2장 주택임대차 상담사례(?:\\[[^\\]]+\\])? 제3장 상가(?:건물)?임대차 상담사례 부록\\s*, 34페이지)"
  ],
  "guide_hlpa_guidebook_2020": [
    "가이드북 시작 러닝헤더 변형 (숫자 마스킹: ^(?:#\\s+)?2020\\. 7\\. 31\\. 개정 주택임대차보호법 가이드북\\s+#(?:\\s+국토교통부-법무부 발간 주택임대차보호법 해설집)?\\s*, 56페이지)",
    "만화 시작 인쇄번호·러닝헤더 (숫자 마스킹: ^#\\s+만화로 보는 주택임(?:대차|차)보호법\\s+[‘\\x22\\x27](?:세입자|집주인)편[’\\x22\\x27]\\s*, 31페이지)",
    "서울시 상담 시작 러닝헤더 (숫자 마스킹: ^#\\s+서울시 주요상담 사례\\s+#\\s+, 5페이지)",
    "부록 시작 인쇄번호·러닝헤더 (숫자 마스킹: ^#\\s+부록\\s+, 10페이지)",
    "뒤섞임·겹자 러닝헤더 (숫자 마스킹: ^\\d{4}\\s+(?:만서화울로시 보주는요 상주담택 임사대례차보호법|만주화택로임 대보차는분 주쟁택조임정대 사차례보호법)\\s+[‘\\x22\\x27]세입자편[’\\x22\\x27](?:\\s+#)?\\s*, 7페이지)"
  ],
  "guide_legal_procedure_seoul": [
    "서울시 견본서식 끝 안내문 (숫자 마스킹: \\s*본 서식은 서울시 주택정책과에서 작성한 견본 문서이므로, 신청인의 구체적인 신청 사유에 맞게 수정하여 사용하시기 바랍니다\\(문의사항: 02-2133-1200~1208\\(3\\)

## 7. 페이지 단위 전처리·metadata 구성

입력 순서를 유지해 페이지별 레코드를 만들고, 제외 사유와 머리말 제거 길이 변화를 함께 기록한다.

In [6]:
DROP_REASONS = ('empty_cover', 'scrambled', 'divider', 'toc')
processed_by_slug: dict[str, list[dict[str, Any]]] = {}
dropped_by_slug: dict[str, dict[str, list[int]]] = {}
header_loss_violations: list[dict[str, Any]] = []
header_applications: dict[str, Counter[str]] = defaultdict(Counter)
document_cleanup_applications: dict[str, Counter[str]] = defaultdict(Counter)
runaway_repetition_applications: list[dict[str, Any]] = []
trailing_printed_number_removed_pages: list[int] = []
trailing_printed_number_mismatches: list[dict[str, int | None]] = []
table_separator_signatures_before: list[tuple[str, int, int, int, tuple[tuple[bool, bool], ...]]] = []

for slug, cfg in DOC_REGISTRY.items():
    output_records: list[dict[str, Any]] = []
    dropped = {reason: [] for reason in DROP_REASONS}
    offset = book_page_info[slug]['offset']
    for row in input_records_by_slug[slug]:
        source_metadata = row['metadata']
        page_index = int(source_metadata['page'])
        cleaned, had_cover_marker = basic_pdf_cleanup(row['page_content'])
        if had_cover_marker and not cleaned:
            dropped['empty_cover'].append(page_index)
            continue
        if is_dotted_leader_toc(cleaned):
            dropped['toc'].append(page_index)
            continue
        if is_divider(cleaned, cfg['doc_title']):
            dropped['divider'].append(page_index)
            continue
        before_boundary_cleanup = cleaned
        applied: list[str] = []
        if legacy_by_slug[slug]:
            cleaned, applied = remove_repeated_boundaries(slug, cleaned)
            header_applications[slug].update(applied)
        if not cleaned or is_divider(cleaned, cfg['doc_title']):
            dropped['divider'].append(page_index)
            continue
        if legacy_by_slug[slug] and before_boundary_cleanup and len(cleaned) / len(before_boundary_cleanup) <= 0.30:
            header_loss_violations.append({'slug': slug, 'page_index': page_index, 'before': len(before_boundary_cleanup), 'after': len(cleaned), 'patterns': applied})
        if legacy_by_slug[slug] and is_scrambled(slug, page_index, cleaned):
            dropped['scrambled'].append(page_index)
            continue
        cleaned, cleanup_counts = apply_document_specific_cleanup(slug, cleaned)
        before_runaway_length = len(cleaned)
        cleaned, runaway_count = collapse_runaway_repetitions(cleaned)
        if runaway_count:
            cleanup_counts['반복 폭주 런'] += runaway_count
            runaway_repetition_applications.append({
                'slug': slug, 'page_index': page_index, 'run_count': runaway_count,
                'before_length': before_runaway_length, 'after_length': len(cleaned),
            })
        if slug == 'guide_hlpa_guidebook_2020' and page_index == 13:
            cleaned = cleaned.replace('•계 약이 묵시적으로 갱신된 경우에는', '•계약이 묵시적으로 갱신된 경우에는')
        document_cleanup_applications[slug].update(cleanup_counts)
        assert cleaned, (slug, page_index)
        pdf_page = page_index + 1
        computed_book_page = None if offset is None else page_index - offset
        book_page = computed_book_page if computed_book_page is not None and computed_book_page >= 1 else None
        if slug == 'counsel_casebook_2024' and (trailing_match := TRAILING_BARE_PAGE_NUMBER_RE.search(cleaned)):
            observed_page_number = int(trailing_match.group(1))
            if book_page is not None and observed_page_number == book_page:
                cleaned = TRAILING_BARE_PAGE_NUMBER_RE.sub('', cleaned).rstrip()
                trailing_printed_number_removed_pages.append(page_index)
            else:
                trailing_printed_number_mismatches.append({'page_index': page_index, 'observed': observed_page_number, 'book_page': book_page})
        cleaned, separator_signatures = normalize_markdown_table_separators(cleaned)
        table_separator_signatures_before.extend((slug, page_index, line_index, column_count, alignment) for line_index, column_count, alignment in separator_signatures)
        load_method = source_metadata['method'] if 'method' in source_metadata else 'pypdf'
        metadata = {
            'source_type': cfg['source_type'],
            'source_id': slug,
            'parent_id': slug,
            'doc_title': cfg['doc_title'],
            'source_org': cfg['source_org'],
            'doc_year': cfg['doc_year'],
            'authority': 'persuasive',
            'stage': 'raw_document',
            'issue': cfg['issue'],
            'source_file': f"data/legal_api_v2/01_raw_pdf/{cfg['raw_pdf']}",
            'load_file': f"data/legal_api_v2/03_load_pdf_json/{cfg['load_json']}",
            'load_method': load_method,
            'pdf_md5': pdf_md5_by_slug[slug],
            'total_pages': int(source_metadata['total_pages']),
            'page_index': page_index,
            'pdf_page': pdf_page,
            'book_page': book_page,
            'record_id': f'{slug}:page:{pdf_page:04d}',
            'section': 'page',
            'content_type': 'text',
            'has_image': False,
            'image_count': 0,
            'images': [],
        }
        output_records.append({'page_content': cleaned, 'metadata': metadata})
    processed_by_slug[slug] = output_records
    dropped_by_slug[slug] = dropped

assert not header_loss_violations, header_loss_violations
print(json.dumps({slug: {'output': len(processed_by_slug[slug]), 'dropped': {reason: len(pages) for reason, pages in dropped_by_slug[slug].items()}} for slug in DOC_REGISTRY}, ensure_ascii=False, indent=2))


{
  "mediation_casebook_2021": {
    "output": 172,
    "dropped": {
      "empty_cover": 23,
      "scrambled": 0,
      "divider": 0,
      "toc": 0
    }
  },
  "mediation_casebook_2022": {
    "output": 137,
    "dropped": {
      "empty_cover": 15,
      "scrambled": 0,
      "divider": 0,
      "toc": 0
    }
  },
  "mediation_casebook_2023": {
    "output": 75,
    "dropped": {
      "empty_cover": 21,
      "scrambled": 0,
      "divider": 0,
      "toc": 0
    }
  },
  "counsel_casebook_2024": {
    "output": 92,
    "dropped": {
      "empty_cover": 20,
      "scrambled": 0,
      "divider": 0,
      "toc": 1
    }
  },
  "counsel_casebook_2025": {
    "output": 103,
    "dropped": {
      "empty_cover": 0,
      "scrambled": 0,
      "divider": 18,
      "toc": 0
    }
  },
  "standard_contract_202310": {
    "output": 5,
    "dropped": {
      "empty_cover": 0,
      "scrambled": 0,
      "divider": 0,
      "toc": 0
    }
  },
  "guide_hlpa_guidebook_2020": {
    "output":

## 8. JSONL 산출물 기록

02 노트북의 안전한 출력 초기화와 compact JSONL 기록 함수를 재사용한다.

In [7]:
def reset_output_directory(path: Path, allowed_parent: Path) -> None:
    resolved = path.resolve()
    if resolved.parent != allowed_parent.resolve():
        raise RuntimeError(f'출력 삭제 안전성 검사 실패: {resolved}')
    if resolved.exists():
        shutil.rmtree(resolved)
    resolved.mkdir(parents=True, exist_ok=True)


def jsonl_parts(records: list[dict[str, Any]]) -> list[list[bytes]]:
    parts: list[list[bytes]] = [[]]
    size = 0
    for record in records:
        encoded = (json.dumps(record, ensure_ascii=False, separators=(',', ':')) + '\n').encode('utf-8')
        if parts[-1] and size + len(encoded) > JSONL_MAX_BYTES:
            parts.append([])
            size = 0
        parts[-1].append(encoded)
        size += len(encoded)
    return parts


def write_jsonl_limited(path: Path, records: list[dict[str, Any]]) -> list[Path]:
    path.parent.mkdir(parents=True, exist_ok=True)
    parts = jsonl_parts(records)
    if len(parts) == 1:
        paths = [path]
    else:
        paths = [path.with_name(f'{path.stem}_part{index:02d}{path.suffix}') for index in range(1, len(parts) + 1)]
    for output_path, lines in zip(paths, parts):
        output_path.write_bytes(b''.join(lines))
    return paths


reset_output_directory(OUTPUT_ROOT, DATA_ROOT)
TEXT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
written_paths: list[Path] = []
for slug in sorted(DOC_REGISTRY):
    paths = write_jsonl_limited(TEXT_OUTPUT_DIR / f'{slug}.jsonl', processed_by_slug[slug])
    assert paths == [TEXT_OUTPUT_DIR / f'{slug}.jsonl'], (slug, paths)
    written_paths.extend(paths)
print(f'JSONL {len(written_paths)}개 기록 완료')


JSONL 10개 기록 완료


## 9. metadata·페이지 보존·HTML·md5 전건 검증

02 노트북 마지막 검증 셀 패턴처럼 전체 산출물을 다시 읽어 모든 완료 조건을 assert하고 validation 리포트를 저장한다.

In [8]:
EXPECTED_METADATA_KEYS = {
    'source_type', 'source_id', 'parent_id', 'doc_title', 'source_org', 'doc_year', 'authority', 'stage', 'issue',
    'source_file', 'load_file', 'load_method', 'pdf_md5', 'total_pages', 'page_index', 'pdf_page', 'book_page',
    'record_id', 'section', 'content_type', 'has_image', 'image_count', 'images',
}


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]


output_by_slug = {slug: read_jsonl(TEXT_OUTPUT_DIR / f'{slug}.jsonl') for slug in DOC_REGISTRY}
all_output_records = [row for slug in DOC_REGISTRY for row in output_by_slug[slug]]
record_ids = [row['metadata']['record_id'] for row in all_output_records]
forbidden_content_ids = [
    row['metadata']['record_id'] for row in all_output_records
    if HTML_COMMENT_RE.search(row['page_content']) or IMAGE_TAG_RE.search(row['page_content'])
    or BREAK_TAG_RE.search(row['page_content']) or BLOCK_TAG_RE.search(row['page_content'])
    or CELL_TAG_RE.search(row['page_content']) or INLINE_TAG_RE.search(row['page_content'])
    or ALLOWLIST_CLOSING_TAG_RE.search(row['page_content'])
]


def page_order_and_relation_ok() -> bool:
    for slug, rows in output_by_slug.items():
        indices = [row['metadata']['page_index'] for row in rows]
        if indices != sorted(indices) or len(indices) != len(set(indices)):
            return False
        if any(row['metadata']['pdf_page'] != row['metadata']['page_index'] + 1 for row in rows):
            return False
    return True


def book_page_consistent() -> bool:
    for slug, rows in output_by_slug.items():
        offset = book_page_info[slug]['offset']
        for row in rows:
            page_index = row['metadata']['page_index']
            expected = None if offset is None or page_index - offset < 1 else page_index - offset
            if row['metadata']['book_page'] != expected:
                return False
    return True


def conservation_ok() -> bool:
    return all(len(input_records_by_slug[slug]) == len(output_by_slug[slug]) + sum(len(pages) for pages in dropped_by_slug[slug].values()) for slug in DOC_REGISTRY)


def compact_jsonl_ok() -> bool:
    for path in sorted(TEXT_OUTPUT_DIR.glob('*.jsonl')):
        for line in path.read_text(encoding='utf-8').splitlines():
            if line != json.dumps(json.loads(line), ensure_ascii=False, separators=(',', ':')):
                return False
    return True


guide_rows = output_by_slug['guide_hlpa_guidebook_2020']
guide_header_residual_ids = [row['metadata']['record_id'] for row in guide_rows if GUIDE_RUNNING_HEADER_RESIDUAL_RE.search(row['page_content'])]
overlapped_run_residual_ids = [row['metadata']['record_id'] for row in output_by_slug['counsel_casebook_2025'] if OVERLAPPED_CHARACTER_RUN_RE.search(row['page_content'])]
runaway_repetition_residual_ids = [row['metadata']['record_id'] for row in all_output_records if RUNAWAY_REPETITION_VALIDATION_RE.search(row['page_content'])]
runaway_repetition_affected_pages = {(item['slug'], item['page_index']) for item in runaway_repetition_applications}
standard_footer_residual_ids = [row['metadata']['record_id'] for row in output_by_slug['standard_contract_202310'] if STANDARD_CONTRACT_FOOTER_RE.search(row['page_content'])]
expected_guide_scrambled_pages = SCRAMBLED_COMIC_PAGES | LEGAL_TEXT_INTERLEAVED_PAGES
actual_guide_scrambled_pages = set(dropped_by_slug['guide_hlpa_guidebook_2020']['scrambled'])
guide_output_page_indices = {row['metadata']['page_index'] for row in guide_rows}
actual_toc_drop_pages = {(slug, page_index) for slug in DOC_REGISTRY for page_index in dropped_by_slug[slug]['toc']}
trailing_number_residuals_2024: list[dict[str, int | None]] = []
for row in output_by_slug['counsel_casebook_2024']:
    if trailing_match := TRAILING_BARE_PAGE_NUMBER_RE.search(row['page_content']):
        trailing_number_residuals_2024.append({'page_index': row['metadata']['page_index'], 'observed': int(trailing_match.group(1)), 'book_page': row['metadata']['book_page']})
matching_trailing_number_residuals_2024 = [item for item in trailing_number_residuals_2024 if item['observed'] == item['book_page']]
table_separator_signatures_after: list[tuple[str, int, int, int, tuple[tuple[bool, bool], ...]]] = []
noncompact_table_separator_ids: list[str] = []
table_separator_count_by_slug: Counter[str] = Counter()
for slug, rows in output_by_slug.items():
    for row in rows:
        lines = row['page_content'].split('\n')
        for line_index in markdown_table_separator_indices(lines):
            column_count, alignment = table_separator_signature(lines[line_index])
            table_separator_signatures_after.append((slug, row['metadata']['page_index'], line_index, column_count, alignment))
            table_separator_count_by_slug[slug] += 1
            if not COMPACT_MARKDOWN_TABLE_SEPARATOR_RE.fullmatch(lines[line_index]):
                noncompact_table_separator_ids.append(f"{row['metadata']['record_id']}:line:{line_index}")


checks = {
    'record_id_global_unique': bool(record_ids) and len(record_ids) == len(set(record_ids)),
    'nonempty_page_content': all(bool(row['page_content'].strip()) for row in all_output_records),
    'html_comments_and_forbidden_tags_absent': not forbidden_content_ids,
    'pdf_page_relation_and_monotonic_page_index': page_order_and_relation_ok(),
    'book_page_offset_consistency': book_page_consistent(),
    'page_record_conservation_by_drop_reason': conservation_ok(),
    'header_removal_no_70pct_loss': not header_loss_violations,
    'guide_running_header_prefix_absent': not guide_header_residual_ids,
    'overlapped_character_runs_absent': not overlapped_run_residual_ids,
    'runaway_repetition_absent': not runaway_repetition_residual_ids,
    'runaway_repetition_only_expected_record': runaway_repetition_affected_pages == {('mediation_casebook_2023', 91)} and sum(item['run_count'] for item in runaway_repetition_applications) == 1,
    'standard_contract_printed_footer_absent': not standard_footer_residual_ids,
    'guide_page_74_restored': 74 in guide_output_page_indices and 74 not in actual_guide_scrambled_pages,
    'guide_scrambled_policy_exact': actual_guide_scrambled_pages == expected_guide_scrambled_pages,
    'toc_drop_only_expected_page': detected_toc_pages == EXPECTED_TOC_PAGES and actual_toc_drop_pages == EXPECTED_TOC_PAGES,
    'no_trailing_printed_page_number_2024': len(trailing_printed_number_removed_pages) == 78 and not matching_trailing_number_residuals_2024,
    'all_table_separator_lines_compact': not noncompact_table_separator_ids,
    'table_count_404_and_column_signatures_preserved': len(table_separator_signatures_before) == EXPECTED_MARKDOWN_TABLE_COUNT and table_separator_signatures_after == table_separator_signatures_before,
    'registry_has_all_unique_documents': len(output_by_slug) == 10 and set(output_by_slug) == set(DOC_REGISTRY),
    'metadata_schema_exact': all(set(row['metadata']) == EXPECTED_METADATA_KEYS for row in all_output_records),
    'metadata_constants_and_md5_match': all(
        row['metadata']['source_id'] == slug and row['metadata']['parent_id'] == slug
        and row['metadata']['pdf_md5'] == pdf_md5_by_slug[slug]
        and row['metadata']['authority'] == 'persuasive' and row['metadata']['stage'] == 'raw_document'
        and row['metadata']['section'] == 'page' and row['metadata']['content_type'] == 'text'
        and row['metadata']['has_image'] is False and row['metadata']['image_count'] == 0 and row['metadata']['images'] == []
        for slug, rows in output_by_slug.items() for row in rows
    ),
    'new_format_md5_matches_raw_pdf': all(legacy_by_slug[slug] or all(row['metadata']['md5'] == pdf_md5_by_slug[slug] for row in input_records_by_slug[slug]) for slug in DOC_REGISTRY),
    'ten_jsonl_files_only': len(list(TEXT_OUTPUT_DIR.glob('*.jsonl'))) == 10 and {path.stem for path in TEXT_OUTPUT_DIR.glob('*.jsonl')} == set(DOC_REGISTRY),
    'compact_utf8_jsonl': compact_jsonl_ok(),
    'excluded_duplicate_inventory_exact': all_loads - selected_loads == set(EXCLUDED_DUPLICATES),
}

documents_report: dict[str, dict[str, Any]] = {}
for slug in DOC_REGISTRY:
    documents_report[slug] = {
        'input_records': len(input_records_by_slug[slug]),
        'records': len(output_by_slug[slug]),
        'dropped': {reason: {'count': len(dropped_by_slug[slug][reason]), 'page_index': dropped_by_slug[slug][reason]} for reason in DROP_REASONS},
        'book_page_offset': book_page_info[slug]['offset'],
        'book_page_method': book_page_info[slug]['method'],
        'book_page_candidate_pages': book_page_info[slug]['candidate_pages'],
        'book_page_matched_pages': book_page_info[slug]['matched_pages'],
        'book_page_agreement': book_page_info[slug]['agreement'],
        'book_page_reason': book_page_info[slug]['reason'],
        'removed_header_patterns': removed_pattern_report.get(slug, []),
        'removed_header_pattern_hits': dict(header_applications.get(slug, Counter())),
        'document_cleanup_pattern_hits': dict(document_cleanup_applications.get(slug, Counter())),
        'runaway_repetition_collapses': [item for item in runaway_repetition_applications if item['slug'] == slug],
        'toc_detected_page_index': sorted(page_index for candidate_slug, page_index in detected_toc_pages if candidate_slug == slug),
        'trailing_printed_page_number_removed': ({'count': len(trailing_printed_number_removed_pages), 'page_index': trailing_printed_number_removed_pages} if slug == 'counsel_casebook_2024' else {'count': 0, 'page_index': []}),
        'trailing_printed_page_number_mismatches': (trailing_printed_number_mismatches if slug == 'counsel_casebook_2024' else []),
        'markdown_table_separator_count': table_separator_count_by_slug[slug],
        'scrambled_policy': ({
            'comic_pages': sorted(SCRAMBLED_COMIC_PAGES),
            'legal_text_interleaved_range': {'start_page_index': min(LEGAL_TEXT_INTERLEAVED_PAGES), 'end_page_index': max(LEGAL_TEXT_INTERLEAVED_PAGES), 'basis': 'original_pdf_visual_review'},
        } if slug == 'guide_hlpa_guidebook_2020' else {}),
        'pdf_md5': pdf_md5_by_slug[slug],
    }

validation_result = {
    'passed': all(checks.values()),
    'checks': checks,
    'documents': documents_report,
    'runaway_repetition_normalization': {'collapsed_run_count': sum(item['run_count'] for item in runaway_repetition_applications), 'affected_records': runaway_repetition_applications, 'residual_record_ids': runaway_repetition_residual_ids},
    'markdown_table_normalization': {'separator_count': len(table_separator_signatures_after), 'expected_count': EXPECTED_MARKDOWN_TABLE_COUNT, 'noncompact_residual': noncompact_table_separator_ids},
    'excluded_duplicates': EXCLUDED_DUPLICATES,
}
VALIDATION_PATH.write_text(json.dumps(validation_result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8', newline='\n')
print(json.dumps(validation_result, ensure_ascii=False, indent=2))
assert validation_result['passed'], {key: value for key, value in checks.items() if not value}


{
  "passed": true,
  "checks": {
    "record_id_global_unique": true,
    "nonempty_page_content": true,
    "html_comments_and_forbidden_tags_absent": true,
    "pdf_page_relation_and_monotonic_page_index": true,
    "book_page_offset_consistency": true,
    "page_record_conservation_by_drop_reason": true,
    "header_removal_no_70pct_loss": true,
    "guide_running_header_prefix_absent": true,
    "overlapped_character_runs_absent": true,
    "runaway_repetition_absent": true,
    "runaway_repetition_only_expected_record": true,
    "standard_contract_printed_footer_absent": true,
    "guide_page_74_restored": true,
    "guide_scrambled_policy_exact": true,
    "toc_drop_only_expected_page": true,
    "no_trailing_printed_page_number_2024": true,
    "all_table_separator_lines_compact": true,
    "table_count_404_and_column_signatures_preserved": true,
    "registry_has_all_unique_documents": true,
    "metadata_schema_exact": true,
    "metadata_constants_and_md5_match": true,
    

## 10. 문서별 완료 요약

출력 레코드 수, 제외 사유별 수, `book_page` 오프셋과 산정 방법을 한 표로 확인한다.

In [9]:
summary_rows = []
for slug, report in documents_report.items():
    drops = report['dropped']
    summary_rows.append({
        'slug': slug,
        'records': report['records'],
        'empty_cover': drops['empty_cover']['count'],
        'scrambled': drops['scrambled']['count'],
        'divider': drops['divider']['count'],
        'toc': drops['toc']['count'],
        'book_page_offset': report['book_page_offset'],
        'method': report['book_page_method'],
    })
summary_rows


[{'slug': 'mediation_casebook_2021',
  'records': 172,
  'empty_cover': 23,
  'scrambled': 0,
  'divider': 0,
  'toc': 0,
  'book_page_offset': 4,
  'method': 'measured_constant_scan'},
 {'slug': 'mediation_casebook_2022',
  'records': 137,
  'empty_cover': 15,
  'scrambled': 0,
  'divider': 0,
  'toc': 0,
  'book_page_offset': 3,
  'method': 'measured_constant_scan'},
 {'slug': 'mediation_casebook_2023',
  'records': 75,
  'empty_cover': 21,
  'scrambled': 0,
  'divider': 0,
  'toc': 0,
  'book_page_offset': 1,
  'method': 'measured_constant_scan'},
 {'slug': 'counsel_casebook_2024',
  'records': 92,
  'empty_cover': 20,
  'scrambled': 0,
  'divider': 0,
  'toc': 1,
  'book_page_offset': 0,
  'method': 'pypdf_boundary_regex_80pct'},
 {'slug': 'counsel_casebook_2025',
  'records': 103,
  'empty_cover': 0,
  'scrambled': 0,
  'divider': 18,
  'toc': 0,
  'book_page_offset': 1,
  'method': 'pypdf_boundary_regex_80pct'},
 {'slug': 'standard_contract_202310',
  'records': 5,
  'empty_cover